# 03 完整QuKerNet搜索链与微调

本节属于课程进阶模块。代码、教学构造、历史实测、待执行实验分别标注。云执行默认关闭；真实金融数据通过仓库外的私有快照接入。原始行情、逐样本标签、角度、核矩阵、参数与预测不得存入公开Notebook输出。

**来源**：[L001 ICLR正式版](https://proceedings.iclr.cc/paper_files/paper/2024/file/4d9944ab3330fe6af8efb9260aa9f307-Paper-Conference.pdf)、[本课程协议](../../docs/天衍云端复现协议.md)。署名：Julius Zhou、AUFE、Cong Lei。代码MIT，原创教学文字与图表CC BY 4.0。


In [1]:
from pathlib import Path
import sys, os, json
import numpy as np
import pandas as pd
COURSE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "课程大纲.md").exists())
MODULE = COURSE / "experiments/qukernet_finance"
sys.path.insert(0, str(MODULE))
PUBLIC = COURSE / "reports/qukernet_cloud_v2"
ENABLE_CLOUD = False
print("默认云执行关闭；当前Notebook不读取凭据、不提交任务。")


默认云执行关闭；当前Notebook不读取凭据、不提交任务。


## 方法链与可比较目标
训练线路KTA标签→MLP→50,000新候选评分→top-k→每布局多配置微调→验证所有终点→选定模型→后续季度评价。50,000个候选的MLP预测是经典计算，不代表50,000次云核矩阵计算。

分类$A=\langle K,T\rangle/(n\|K\|_F)$，$T_{ij}=2\mathbf1[y_i=y_j]-1$。回归先中心化核与标签。训练KTA、验证准确率、季度准确率对应不同问题。

In [2]:
from cloud_v2.circuits import sample_layout, valid_positions, encode_layout, budget_estimate
layout=sample_layout(np.random.default_rng(20261110),8,40,(1,2,3,4,5))
print("线路比特:",layout["qubits"],"特征:",layout["features"],"可替换的真实编码位置:",len(valid_positions(layout)))
print("评分器编码向量维度:",encode_layout(layout,5).size)
estimate=budget_estimate(n=16,v=16,o=220,labels=16,top_k=2,configurations=2,steps=3,batch_size=8,max_parameters=2,shots=128)
display(pd.Series(estimate).drop("assumptions"))

线路比特: 8 特征: 40 可替换的真实编码位置: 160
评分器编码向量维度: 2000


label_probabilities                        1920
preselection_probabilities                  752
finetuning_probabilities                   5008
final_test_probabilities                   3520
probability_circuits_per_arm_seed         11200
nominal_shots_per_arm_seed              1433600
probability_circuits_all_repetitions      11200
nominal_shots_all_repetitions           1433600
dtype: object

## 梯度不能对整个KTA直接移位
对RX/RY/RZ单个物理门出现，$\partial p/\partial\phi=[p(\phi+\pi/2)-p(\phi-\pi/2)]/2$。共享参数在$U(x)$和$U(x')^\dagger$两端出现；逐出现求导、计入逆门符号，再通过KTA归一化的链式法则相加。直接计算$[A(\theta+\pi/2)-A(\theta-\pi/2)]/2$通常不成立。

替换位置、参数和特征顺序必须绑定保存。所有配置终点参与V选择，不能先以训练KTA淘汰到一条后把它称完整验证选择。

In [3]:
# 解析构造：核两端共同的最后单比特旋转可能完全抵消。
def toy_kernel(a,b,theta):
    return np.cos(((a+theta)-(b+theta))/2)**2
grid=np.array([toy_kernel(.2,1.1,t) for t in [-.7,0.,.8]])
assert np.allclose(grid,grid[0])
print("共同旋转抵消后的核:",grid)
print("零梯度不必是程序错误；须检查是否结构性退化。")

共同旋转抵消后的核: [0.81080498 0.81080498 0.81080498]
零梯度不必是程序错误；须检查是否结构性退化。


## 运行与验收
`experiments --dry-run`只核算资源；`client_entry --execute`才尝试云执行，仍须通过费用与能力检查。未满足条件时，本课程不把本地概率替代品写成云结果。

**作业**：对一个2比特、两个参数的构造线路比较逐门参数移位和中心有限差分；证明重载参数与mask后结果不变。记录全部科学失效配置，并说明它是否减少了最终实际比较数量。